In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import nibabel as nib
from scipy import ndimage

# =========================================================
# PATHS
# =========================================================

UNEST_ROOT = "path/to/download_file"
SYNTH_ROOT = "path/to/IXI_SYNTHSEG_BATCH"

OUT = "path/to/UNesT_SynthSeg_spatial_statistics.csv"


# =========================================================
# SYNTHSEG LABEL -> CANONICAL
# =========================================================

synth_map = {
    2:  ("BRN-CER-WM", "LEFT"),
    4:  ("BRN-LAT-V", "LEFT"),
    5:  ("BRN-INF-LAT-V", "LEFT"),
    7:  ("BRN-CB-WM", "LEFT"),
    8:  ("BRN-CB-EXT", "LEFT"),
    10: ("BRN-THA", "LEFT"),
    11: ("BRN-CAU", "LEFT"),
    12: ("BRN-PUT", "LEFT"),
    13: ("BRN-PAL", "LEFT"),
    14: ("BRN-3RD-VENTRICLE", "NONE"),
    15: ("BRN-4TH-VENTRICLE", "NONE"),
    16: ("BRN-BST", "NONE"),
    17: ("BRN-HIP", "LEFT"),
    18: ("BRN-AMY", "LEFT"),
    26: ("BRN-ACC", "LEFT"),
    28: ("BRN-VDC", "LEFT"),

    41: ("BRN-CER-WM", "RIGHT"),
    43: ("BRN-LAT-V", "RIGHT"),
    44: ("BRN-INF-LAT-V", "RIGHT"),
    46: ("BRN-CB-WM", "RIGHT"),
    47: ("BRN-CB-EXT", "RIGHT"),
    49: ("BRN-THA", "RIGHT"),
    50: ("BRN-CAU", "RIGHT"),
    51: ("BRN-PUT", "RIGHT"),
    52: ("BRN-PAL", "RIGHT"),
    53: ("BRN-HIP", "RIGHT"),
    54: ("BRN-AMY", "RIGHT"),
    58: ("BRN-ACC", "RIGHT"),
    60: ("BRN-VDC", "RIGHT"),
}


canonical_to_synth = {
    v: k for k, v in synth_map.items()
}


# =========================================================
# METRICS
# =========================================================

def dice(a, b):
    a = a.astype(bool)
    b = b.astype(bool)

    inter = np.logical_and(a, b).sum()
    total = a.sum() + b.sum()

    if total == 0:
        return 1.0

    return 2.0 * inter / total


def iou(a, b):
    a = a.astype(bool)
    b = b.astype(bool)

    inter = np.logical_and(a, b).sum()
    union = np.logical_or(a, b).sum()

    if union == 0:
        return 1.0

    return inter / union


def surface_distances(a, b, spacing=(1, 1, 1)):

    a = a.astype(bool)
    b = b.astype(bool)

    if not a.any() or not b.any():
        return None, None

    structure = ndimage.generate_binary_structure(3, 1)

    a_surface = a ^ ndimage.binary_erosion(
        a,
        structure=structure,
        border_value=0
    )

    b_surface = b ^ ndimage.binary_erosion(
        b,
        structure=structure,
        border_value=0
    )

    dt_a = ndimage.distance_transform_edt(
        ~a_surface,
        sampling=spacing
    )

    dt_b = ndimage.distance_transform_edt(
        ~b_surface,
        sampling=spacing
    )

    d_ab = dt_b[a_surface]
    d_ba = dt_a[b_surface]

    return d_ab, d_ba


def hd95(a, b, spacing=(1, 1, 1)):

    d_ab, d_ba = surface_distances(
        a, b, spacing
    )

    if d_ab is None:
        return np.nan

    return max(
        np.percentile(d_ab, 95),
        np.percentile(d_ba, 95)
    )


def assd(a, b, spacing=(1, 1, 1)):

    d_ab, d_ba = surface_distances(
        a, b, spacing
    )

    if d_ab is None:
        return np.nan

    return (
        d_ab.sum() + d_ba.sum()
    ) / (
        len(d_ab) + len(d_ba)
    )



# =========================================================
# FIND SUBJECTS
# =========================================================

unest_files = glob.glob(
    os.path.join(
        UNEST_ROOT,
        "*",
        "MNI305",
        "*_UNEST_segmentation_MNI305.nii.gz"
    )
)

synth_files = glob.glob(
    os.path.join(
        SYNTH_ROOT,
        "*",
        "MNI305",
        "*_SynthSeg_segmentation_MNI305.nii.gz"
    )
)

unest_dict = {
    os.path.basename(f).split("_")[0]: f
    for f in unest_files
}

synth_dict = {
    os.path.basename(f).split("_")[0]: f
    for f in synth_files
}

subjects = sorted(
    set(unest_dict) & set(synth_dict)
)

print("UNesT subjects:", len(unest_dict))
print("SynthSeg subjects:", len(synth_dict))
print("Common subjects:", len(subjects))


# =========================================================
# CALCULATE SPATIAL METRICS
# =========================================================

results = []

for i, subject_id in enumerate(subjects, 1):

    print(
        f"[{i}/{len(subjects)}] {subject_id}"
    )

    unest_img = nib.load(
        unest_dict[subject_id]
    )

    synth_img = nib.load(
        synth_dict[subject_id]
    )

    unest = np.asarray(
        unest_img.dataobj
    ).astype(np.int16)

    synth = np.asarray(
        synth_img.dataobj
    ).astype(np.int16)

    # -----------------------------------------------------
    # Verify common grid
    # -----------------------------------------------------

    if unest.shape != synth.shape:
        print(
            "  SKIP: shape mismatch",
            unest.shape,
            synth.shape
        )
        continue

    spacing = tuple(
        unest_img.header.get_zooms()[:3]
    )

    # -----------------------------------------------------
    # Each canonical structure
    # -----------------------------------------------------

    for (canonical_id, laterality), synth_label in canonical_to_synth.items():

        # UNesT source IDs for this canonical concept
        # -------------------------------------------------
        # Mapping is inferred from canonical mapping CSV
        # using the existing UNesT source mappings.

        # Load only once
        pass

In [ ]:
# =========================================================
# LOAD UNEST MAPPING
# =========================================================

MAPPING_CSV = ("path/to/source_mapping_MONAI_UNEST_v0.2.7_FINAL.csv")

mapping = pd.read_csv(MAPPING_CSV)

mapping["source_id"] = (
    mapping["source_id"]
    .astype(int)
)

unest_groups = (
    mapping[
        mapping["canonical_id"].isin(
            [x[0] for x in canonical_to_synth.keys()]
        )
    ]
    .groupby(
        ["canonical_id", "laterality"]
    )["source_id"]
    .apply(list)
    .to_dict()
)

In [ ]:
# =========================================================
# CALCULATE SPATIAL METRICS
# =========================================================

results = []
for i, subject_id in enumerate(subjects, 1):

    print(f"[{i}/{len(subjects)}] {subject_id}")

    unest_img = nib.load(unest_dict[subject_id])
    synth_img = nib.load(synth_dict[subject_id])

    unest = np.asarray(
        unest_img.dataobj
    ).astype(np.int16)

    synth = np.asarray(
        synth_img.dataobj
    ).astype(np.int16)

    if unest.shape != synth.shape:
        print("  SKIP shape mismatch")
        continue

    spacing = tuple(
        unest_img.header.get_zooms()[:3]
    )

    for (canonical_id, laterality), synth_label in canonical_to_synth.items():

        unest_labels = unest_groups.get(
            (canonical_id, laterality),
            []
        )

        if not unest_labels:
            continue

        # -------------------------------------------------
        # Build canonical masks
        # -------------------------------------------------

        unest_mask = np.isin(
            unest,
            unest_labels
        )

        synth_mask = (
            synth == synth_label
        )

        # -------------------------------------------------
        # Metrics
        # -------------------------------------------------

        d = dice(
            unest_mask,
            synth_mask
        )

        j = iou(
            unest_mask,
            synth_mask
        )

        h = hd95(
            unest_mask,
            synth_mask,
            spacing
        )

        a = assd(
            unest_mask,
            synth_mask,
            spacing
        )

        results.append({
            "subject_id": subject_id,
            "canonical_id": canonical_id,
            "laterality": laterality,
            "Dice": d,
            "IoU": j,
            "HD95_mm": h,
            "ASSD_mm": a,
            "UNesT_voxels": int(unest_mask.sum()),
            "SynthSeg_voxels": int(synth_mask.sum())
        })


spatial_df = pd.DataFrame(results)

spatial_df.to_csv(
    OUT,
    index=False
)

print("\nSaved:")
print(OUT)

print("\nRows:", len(spatial_df))
print("Subjects:", spatial_df.subject_id.nunique())
print(
    "Structures:",
    spatial_df[
        ["canonical_id", "laterality"]
    ].drop_duplicates().shape[0]
)

display(spatial_df.head())

In [ ]:
summary = (
    spatial_df
    .groupby(
        ["canonical_id", "laterality"]
    )
    .agg(
        n=("subject_id", "count"),

        Dice_mean=("Dice", "mean"),
        Dice_sd=("Dice", "std"),

        IoU_mean=("IoU", "mean"),
        IoU_sd=("IoU", "std"),

        HD95_mean_mm=("HD95_mm", "mean"),
        HD95_sd_mm=("HD95_mm", "std"),

        ASSD_mean_mm=("ASSD_mm", "mean"),
        ASSD_sd_mm=("ASSD_mm", "std"),
    )
    .reset_index()
)

display(
    summary.sort_values(
        "Dice_mean",
        ascending=False
    )
)

In [ ]:
print("=== OVERALL SPATIAL AGREEMENT ===")

for metric in [
    "Dice",
    "IoU",
    "HD95_mm",
    "ASSD_mm"
]:
    print(
        f"{metric}: "
        f"mean={spatial_df[metric].mean():.4f}, "
        f"median={spatial_df[metric].median():.4f}, "
        f"SD={spatial_df[metric].std():.4f}"
    )

In [ ]:
# Overall
print("=== OVERALL ===")
print(spatial_df[["Dice", "IoU", "HD95_mm", "ASSD_mm"]].describe())

# Per structure
summary = (
    spatial_df
    .groupby(["canonical_id", "laterality"])
    .agg(
        n=("subject_id", "count"),
        Dice_mean=("Dice", "mean"),
        Dice_sd=("Dice", "std"),
        IoU_mean=("IoU", "mean"),
        IoU_sd=("IoU", "std"),
        HD95_mean_mm=("HD95_mm", "mean"),
        HD95_sd_mm=("HD95_mm", "std"),
        ASSD_mean_mm=("ASSD_mm", "mean"),
        ASSD_sd_mm=("ASSD_mm", "std"),
    )
    .reset_index()
)

display(summary.sort_values("Dice_mean", ascending=False))

In [ ]:
summary.to_csv(
    "./UNesT_SynthSeg_spatial_summary.csv",
    index=False
)

print("Saved.")

In [ ]:
print("Subjects:", spatial_df.subject_id.nunique())
print(
    "Structure/laterality pairs:",
    spatial_df[["canonical_id", "laterality"]]
    .drop_duplicates()
    .shape[0]
)

print("\nMissing metrics:")
print(spatial_df[["Dice", "IoU", "HD95_mm", "ASSD_mm"]].isna().sum())

In [ ]:
import os
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests

# ============================================================
# PATHS
# ============================================================

UNEST_CSV = "path/to/download_file/UNesT_structure_measurements_ALL.csv"
SYNTHSEG_CSV = "path/to/IXI_SYNTHSEG_BATCH/SynthSeg_structure_measurements_ALL.csv"

OUTPUT_CSV = "./UNesT_SynthSeg_Volumetric_Agreement_demo.csv"

# ============================================================
# LOAD DATA
# ============================================================

unest = pd.read_csv(UNEST_CSV)
synthseg = pd.read_csv(SYNTHSEG_CSV)

print("UNesT:", unest.shape)
print("SynthSeg:", synthseg.shape)

# ============================================================
# KEEP ONLY REQUIRED COLUMNS
# ============================================================

u = unest[
    [
        "subject_id",
        "canonical_id",
        "preferred_name",
        "laterality",
        "volume_mm3"
    ]
].copy()

s = synthseg[
    [
        "subject_id",
        "canonical_id",
        "preferred_name",
        "laterality",
        "volume_mm3"
    ]
].copy()

# Rename volume columns

u = u.rename(columns={
    "volume_mm3": "UNesT_volume_mm3"
})

s = s.rename(columns={
    "volume_mm3": "SynthSeg_volume_mm3"
})

# ============================================================
# FIND COMMON CANONICAL STRUCTURES
# ============================================================

common_structures = sorted(
    set(u["canonical_id"].unique())
    &
    set(s["canonical_id"].unique())
)

print("\nCommon canonical structures:")
for x in common_structures:
    print(x)

print("\nNumber of common structures:", len(common_structures))

# ============================================================
# MERGE UNesT + SYNTHSEG
# ============================================================

comparison = pd.merge(
    u,
    s,
    on=[
        "subject_id",
        "canonical_id",
        "laterality"
    ],
    how="inner",
    suffixes=("_UNesT", "_SynthSeg")
)

# Use preferred name from UNesT
comparison["preferred_name"] = comparison["preferred_name_UNesT"]

comparison = comparison[
    [
        "subject_id",
        "canonical_id",
        "preferred_name",
        "laterality",
        "UNesT_volume_mm3",
        "SynthSeg_volume_mm3"
    ]
]

print("\nPaired observations:", len(comparison))

# ============================================================
# BASIC VOLUME DIFFERENCES
# ============================================================

comparison["absolute_difference_mm3"] = (
    comparison["UNesT_volume_mm3"]
    -
    comparison["SynthSeg_volume_mm3"]
)

comparison["absolute_difference_magnitude_mm3"] = (
    comparison["absolute_difference_mm3"].abs()
)

# Percentage difference relative to SynthSeg

comparison["percentage_difference"] = (
    comparison["absolute_difference_mm3"]
    /
    comparison["SynthSeg_volume_mm3"]
) * 100

# Mean volume

comparison["mean_volume_mm3"] = (
    comparison["UNesT_volume_mm3"]
    +
    comparison["SynthSeg_volume_mm3"]
) / 2

# ============================================================
# ICC(2,1)
# Two-way random effects, absolute agreement, single measures
# ============================================================

def icc_2_1(x, y):

    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    data = np.column_stack([x, y])

    n, k = data.shape

    grand_mean = data.mean()

    # Row means = subject means
    row_means = data.mean(axis=1)

    # Column means = method means
    col_means = data.mean(axis=0)

    # Between-subject sum of squares
    SSR = k * np.sum(
        (row_means - grand_mean) ** 2
    )

    # Between-rater sum of squares
    SSC = n * np.sum(
        (col_means - grand_mean) ** 2
    )

    # Residual
    SSE = np.sum(
        (data
         - row_means[:, None]
         - col_means[None, :]
         + grand_mean) ** 2
    )

    MSR = SSR / (n - 1)
    MSC = SSC / (k - 1)
    MSE = SSE / ((n - 1) * (k - 1))

    # ICC(2,1)
    ICC = (
        (MSR - MSE)
        /
        (
            MSR
            + (k - 1) * MSE
            + (k * (MSC - MSE) / n)
        )
    )

    return ICC


# ============================================================
# CALCULATE STRUCTURE-LEVEL STATISTICS
# ============================================================

results = []

group_cols = [
    "canonical_id",
    "preferred_name",
    "laterality"
]

for (canonical_id, preferred_name, laterality), group in comparison.groupby(
    group_cols,
    dropna=False
):

    x = group["UNesT_volume_mm3"].values
    y = group["SynthSeg_volume_mm3"].values

    # --------------------------------------------------------
    # Remove missing / invalid values
    # --------------------------------------------------------

    valid = (
        np.isfinite(x)
        &
        np.isfinite(y)
    )

    x = x[valid]
    y = y[valid]

    n = len(x)

    if n < 3:
        continue

    # --------------------------------------------------------
    # Pearson correlation
    # --------------------------------------------------------

    pearson_r, pearson_p = stats.pearsonr(x, y)

    # --------------------------------------------------------
    # Spearman correlation
    # --------------------------------------------------------

    spearman_rho, spearman_p = stats.spearmanr(x, y)

    # --------------------------------------------------------
    # ICC
    # --------------------------------------------------------

    icc = icc_2_1(x, y)

    # --------------------------------------------------------
    # Bland-Altman
    # --------------------------------------------------------

    difference = x - y

    bias = np.mean(difference)

    sd_difference = np.std(
        difference,
        ddof=1
    )

    loa_lower = bias - 1.96 * sd_difference
    loa_upper = bias + 1.96 * sd_difference

    # --------------------------------------------------------
    # Mean absolute difference
    # --------------------------------------------------------

    mean_abs_difference = np.mean(
        np.abs(difference)
    )

    # --------------------------------------------------------
    # Mean percentage difference
    # --------------------------------------------------------

    mean_percentage_difference = np.mean(
        difference / y * 100
    )

    # --------------------------------------------------------
    # Relative absolute difference
    # --------------------------------------------------------

    mean_absolute_percentage_difference = np.mean(
        np.abs(difference / y) * 100
    )

    # --------------------------------------------------------
    # Proportional bias
    #
    # Difference vs mean volume
    # --------------------------------------------------------

    mean_volume = (x + y) / 2

    slope, intercept, r_value, p_value, std_err = (
        stats.linregress(
            mean_volume,
            difference
        )
    )

    results.append({

        "canonical_id": canonical_id,

        "preferred_name": preferred_name,

        "laterality": laterality,

        "N": n,

        "UNesT_mean_volume_mm3": np.mean(x),

        "SynthSeg_mean_volume_mm3": np.mean(y),

        "UNesT_median_volume_mm3": np.median(x),

        "SynthSeg_median_volume_mm3": np.median(y),

        "ICC_2_1": icc,

        "Pearson_r": pearson_r,

        "Pearson_p": pearson_p,

        "Spearman_rho": spearman_rho,

        "Spearman_p": spearman_p,

        "BA_bias_mm3": bias,

        "BA_LOA_lower_mm3": loa_lower,

        "BA_LOA_upper_mm3": loa_upper,

        "mean_absolute_difference_mm3":
            mean_abs_difference,

        "mean_percentage_difference":
            mean_percentage_difference,

        "mean_absolute_percentage_difference":
            mean_absolute_percentage_difference,

        "proportional_bias_slope":
            slope,

        "proportional_bias_p":
            p_value

    })


results_df = pd.DataFrame(results)

# ============================================================
# FDR CORRECTION
# ============================================================

# Pearson
results_df["Pearson_q"] = multipletests(
    results_df["Pearson_p"],
    method="fdr_bh"
)[1]

# Spearman
results_df["Spearman_q"] = multipletests(
    results_df["Spearman_p"],
    method="fdr_bh"
)[1]

# Proportional bias
results_df["proportional_bias_q"] = multipletests(
    results_df["proportional_bias_p"],
    method="fdr_bh"
)[1]

# ============================================================
# SORT BY ICC
# ============================================================

results_df = results_df.sort_values(
    "ICC_2_1",
    ascending=False
).reset_index(drop=True)

# ============================================================
# SAVE
# ============================================================

results_df.to_csv(
    OUTPUT_CSV,
    index=False
)

print("\nSaved:")
print(OUTPUT_CSV)

# ============================================================
# DISPLAY
# ============================================================

pd.set_option(
    "display.max_columns",
    None
)

display(results_df)

In [ ]:
paper_table = results_df[
    [
        "canonical_id",
        "preferred_name",
        "laterality",
        "N",
        "UNesT_mean_volume_mm3",
        "SynthSeg_mean_volume_mm3",
        "ICC_2_1",
        "Pearson_r",
        "Spearman_rho",
        "BA_bias_mm3",
        "BA_LOA_lower_mm3",
        "BA_LOA_upper_mm3",
        "proportional_bias_q"
    ]
].copy()

paper_table = paper_table.rename(columns={
    "canonical_id": "Canonical ID",
    "preferred_name": "Structure",
    "laterality": "Laterality",
    "N": "N",
    "UNesT_mean_volume_mm3": "UNesT Mean (mm³)",
    "SynthSeg_mean_volume_mm3": "SynthSeg Mean (mm³)",
    "ICC_2_1": "ICC(2,1)",
    "Pearson_r": "Pearson r",
    "Spearman_rho": "Spearman ρ",
    "BA_bias_mm3": "Bias (mm³)",
    "BA_LOA_lower_mm3": "LoA Lower (mm³)",
    "BA_LOA_upper_mm3": "LoA Upper (mm³)",
    "proportional_bias_q":"Proportional-bias q"
})

paper_table.to_csv(
    "./Volumetric_Agreement_Table.csv",
    index=False
)

display(paper_table)

In [ ]:
import os
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests

# ============================================================
# PATHS
# ============================================================

UNEST_CSV = "path/to/download_file/UNesT_structure_measurements_ALL.csv"
SYNTHSEG_CSV = "path/to/IXI_SYNTHSEG_BATCH/SynthSeg_structure_measurements_ALL.csv"

OUTPUT_CSV = "./UNesT_SynthSeg_Volumetric_Agreement_paper.csv"

# ============================================================
# LOAD DATA
# ============================================================

unest = pd.read_csv(UNEST_CSV)
synthseg = pd.read_csv(SYNTHSEG_CSV)

print("UNesT:", unest.shape)
print("SynthSeg:", synthseg.shape)

# ============================================================
# KEEP ONLY REQUIRED COLUMNS
# ============================================================

u = unest[
    [
        "subject_id",
        "canonical_id",
        "preferred_name",
        "laterality",
        "volume_mm3"
    ]
].copy()

s = synthseg[
    [
        "subject_id",
        "canonical_id",
        "preferred_name",
        "laterality",
        "volume_mm3"
    ]
].copy()

# Rename volume columns

u = u.rename(columns={
    "volume_mm3": "UNesT_volume_mm3"
})

s = s.rename(columns={
    "volume_mm3": "SynthSeg_volume_mm3"
})

# ============================================================
# FIND COMMON CANONICAL STRUCTURES
# ============================================================

common_structures = sorted(
    set(u["canonical_id"].unique())
    &
    set(s["canonical_id"].unique())
)

print("\nCommon canonical structures:")
for x in common_structures:
    print(x)

print("\nNumber of common structures:", len(common_structures))

# ============================================================
# MERGE UNesT + SYNTHSEG
# ============================================================

comparison = pd.merge(
    u,
    s,
    on=[
        "subject_id",
        "canonical_id",
        "laterality"
    ],
    how="inner",
    suffixes=("_UNesT", "_SynthSeg")
)

# Use preferred name from UNesT
comparison["preferred_name"] = comparison["preferred_name_UNesT"]

comparison = comparison[
    [
        "subject_id",
        "canonical_id",
        "preferred_name",
        "laterality",
        "UNesT_volume_mm3",
        "SynthSeg_volume_mm3"
    ]
]

print("\nPaired observations:", len(comparison))

# ============================================================
# BASIC VOLUME DIFFERENCES
# ============================================================

comparison["absolute_difference_mm3"] = (
    comparison["UNesT_volume_mm3"]
    -
    comparison["SynthSeg_volume_mm3"]
)

comparison["absolute_difference_magnitude_mm3"] = (
    comparison["absolute_difference_mm3"].abs()
)

# Percentage difference relative to SynthSeg

comparison["percentage_difference"] = (
    comparison["absolute_difference_mm3"]
    /
    comparison["SynthSeg_volume_mm3"]
) * 100

# Mean volume

comparison["mean_volume_mm3"] = (
    comparison["UNesT_volume_mm3"]
    +
    comparison["SynthSeg_volume_mm3"]
) / 2

# ============================================================
# ICC(2,1)
# Two-way random effects, absolute agreement, single measures
# ============================================================

def icc_2_1(x, y):

    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    data = np.column_stack([x, y])

    n, k = data.shape

    grand_mean = data.mean()

    # Row means = subject means
    row_means = data.mean(axis=1)

    # Column means = method means
    col_means = data.mean(axis=0)

    # Between-subject sum of squares
    SSR = k * np.sum(
        (row_means - grand_mean) ** 2
    )

    # Between-rater sum of squares
    SSC = n * np.sum(
        (col_means - grand_mean) ** 2
    )

    # Residual
    SSE = np.sum(
        (data
         - row_means[:, None]
         - col_means[None, :]
         + grand_mean) ** 2
    )

    MSR = SSR / (n - 1)
    MSC = SSC / (k - 1)
    MSE = SSE / ((n - 1) * (k - 1))

    # ICC(2,1)
    ICC = (
        (MSR - MSE)
        /
        (
            MSR
            + (k - 1) * MSE
            + (k * (MSC - MSE) / n)
        )
    )

    return ICC


# ============================================================
# CALCULATE STRUCTURE-LEVEL STATISTICS
# ============================================================

results = []

group_cols = [
    "canonical_id",
    "preferred_name",
    "laterality"
]

for (canonical_id, preferred_name, laterality), group in comparison.groupby(
    group_cols,
    dropna=False
):

    x = group["UNesT_volume_mm3"].values
    y = group["SynthSeg_volume_mm3"].values

    # --------------------------------------------------------
    # Remove missing / invalid values
    # --------------------------------------------------------

    valid = (
        np.isfinite(x)
        &
        np.isfinite(y)
    )

    x = x[valid]
    y = y[valid]

    n = len(x)

    if n < 3:
        continue

    # --------------------------------------------------------
    # Pearson correlation
    # --------------------------------------------------------

    pearson_r, pearson_p = stats.pearsonr(x, y)

    # --------------------------------------------------------
    # Spearman correlation
    # --------------------------------------------------------

    spearman_rho, spearman_p = stats.spearmanr(x, y)

    # --------------------------------------------------------
    # ICC
    # --------------------------------------------------------

    icc = icc_2_1(x, y)

    # --------------------------------------------------------
    # Bland-Altman
    # --------------------------------------------------------

    difference = x - y

    bias = np.mean(difference)

    sd_difference = np.std(
        difference,
        ddof=1
    )

    loa_lower = bias - 1.96 * sd_difference
    loa_upper = bias + 1.96 * sd_difference

    # --------------------------------------------------------
    # Mean absolute difference
    # --------------------------------------------------------

    mean_abs_difference = np.mean(
        np.abs(difference)
    )

    # --------------------------------------------------------
    # Mean percentage difference
    # --------------------------------------------------------

    mean_percentage_difference = np.mean(
        difference / y * 100
    )

    # --------------------------------------------------------
    # Relative absolute difference
    # --------------------------------------------------------

    mean_absolute_percentage_difference = np.mean(
        np.abs(difference / y) * 100
    )

    # --------------------------------------------------------
    # Proportional bias
    #
    # Difference vs mean volume
    # --------------------------------------------------------

    mean_volume = (x + y) / 2

    slope, intercept, r_value, p_value, std_err = (
        stats.linregress(
            mean_volume,
            difference
        )
    )

    results.append({
        "Structure": preferred_name +" "+ laterality[0],


        "N": n,

        "ICC_2_1": icc,

        "Pearson_r": pearson_r,
        
        "Spearman_rho": spearman_rho,
        
        "Spearman_p": spearman_p,

        "Pearson_p": pearson_p,

        "BA_bias_mm3": bias,

        "95% LOA": [loa_lower, loa_upper],

        "BA_LOA_upper_mm3": loa_upper,

        "proportional_bias_p":
            p_value

    })


results_df = pd.DataFrame(results)

# ============================================================
# FDR CORRECTION
# ============================================================

# Pearson
results_df["Pearson_q"] = multipletests(
    results_df["Pearson_p"],
    method="fdr_bh"
)[1]

# Spearman
results_df["Spearman_q"] = multipletests(
    results_df["Spearman_p"],
    method="fdr_bh"
)[1]

# Proportional bias
results_df["proportional_bias_q"] = multipletests(
    results_df["proportional_bias_p"],
    method="fdr_bh"
)[1]
results_df["95% LOA"] = results_df["95% LOA"].apply(
    lambda x: [round(v, 2) for v in x] if isinstance(x, list) else x
)
# ============================================================
# SORT BY ICC
# ============================================================

results_df = results_df.sort_values(
    "ICC_2_1",
    ascending=False
).reset_index(drop=True)

# ============================================================
# SAVE
# ============================================================

results_df.to_csv(
    OUTPUT_CSV,
    index=False
)

print("\nSaved:")
print(OUTPUT_CSV)

# ============================================================
# DISPLAY
# ============================================================

pd.set_option(
    "display.max_columns",
    None
)

display(results_df)